# Trabalho 1 — Aquisição de Dados

## 1. Identificação do projeto

**Tema:**

Análise da relação entre a variação dos preços da cesta básica, a inflação dos alimentos e os diferentes períodos de governos presidenciais e estaduais no Brasil desde 1994.

**Integrantes:**

- Alonso Ramos de Brito Neto - 22552380
- Diego Gabriel Silva Azevedo - 22551909
- Luiz Henrique Barbosa Costa - 22351234
- Adrielly Silva de Souza - 22551157

**Disciplina:** Ciência de Dados

**Instituição:** Universidade Federal do Amazonas — Instituto de Computação

## 2. Pergunta motivadora

De que maneira a variação nos preços da cesta básica e a inflação dos alimentos se
comportaram ao longo das diferentes gestões presidenciais e estaduais no Brasil
desde 1994, e como os ciclos eleitorais e espectros partidários se correlacionam com
esses momentos de instabilidade?

## 3. Objetivo da coleta

Construir uma base de dados integrada contendo informações sobre
a inflação dos alimentos, os preços da cesta básica e informações
relacionadas aos períodos de governos e eleições no Brasil.

A base será utilizada nas etapas posteriores do projeto para
investigar possíveis padrões entre as variáveis econômicas e
políticas.

## 4. Bibliotecas e configurações

### 4.1 Bibliotecas utilizadas

| Biblioteca | Finalidade | Instalação |
|---|---|---|
| `requests` | Requisições HTTP para a API do IBGE/SIDRA e tratamento de status HTTP | `pip install requests` |
| `pandas` | Leitura, limpeza, transformação e integração de dados; leitura de HTML com `read_html` | `pip install pandas` |
| `beautifulsoup4` | Parsing e extração de dados do HTML da página do DIEESE | `pip install beautifulsoup4` |
| `lxml` | Parser HTML/XML de alta performance (motor alternativo para BeautifulSoup) | `pip install lxml` |
| `zipfile` (padrão) | Extração de arquivos CSV de arquivos ZIP baixados do TSE (CDN) | - |
| `io` (padrão) | Manipulação de bytes em memória durante o download/decompresão | - |
| `pyarrow` | Leitura e escrita de arquivos Parquet (formato recomendado para a base tratada) | `pip install pyarrow` |
| `json` (padrão) | Processamento de respostas JSON da API | - |
| `datetime` (padrão) | Registro de data e hora das coletas com fuso horário | - |
| `time` (padrão) | Pausas entre requisições para respeitar limites dos servidores | - |
| `pathlib` (padrão) | Construção de caminhos de arquivos de forma portável | - |

### 4.2 Configurações iniciais

- **User-Agent:** identificar o robô em requisições HTTP, conforme recomendado em `robots.txt`.
- **Timeout:** definir limite de espera por requisição (ex: 30 segundos).
- **Pausas entre requisições:** intervalo de pelo menos 1 segundo entre chamadas à API e entre páginas scrapeadas.
- **Codificação de resposta:** forçar `utf-8` ao ler HTML ou JSON para evitar problemas de acentuação.
- **Seed de data/hora:** registrar cada coleta com `datetime.now(timezone.utc)` ou fuso horário de Brasilia (`America/Sao_Paulo`).


In [3]:
# Bibliotecas e configurações
import requests
import csv
import re
import pandas as pd
import json
import os
import io
import zipfile
from datetime import datetime, timezone
from pathlib import Path
import time

# Parser HTML
from bs4 import BeautifulSoup

# Exportacao Parquet
import pdfplumber
import pyarrow as pa
import pyarrow.parquet as pq

# Configuracoes gerais
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Fuso horario de referencia (Brasilia)
FUSO = "America/Sao_Paulo"

# Caminhos base (ajuste conforme a estrutura da sua maquina)
BASE_DIR = Path.cwd()
WORKSPACE_DIR = BASE_DIR.parent if BASE_DIR.name == "projeto" else BASE_DIR
BRUTOS_DIR = WORKSPACE_DIR / "projeto" / "dados_brutos"
TRATADOS_DIR = WORKSPACE_DIR / "projeto" / "dados_tratados"
DOC_DIR = WORKSPACE_DIR / "projeto" / "documentacao"

# Criar pastas se nao existirem
BRUTOS_DIR.mkdir(parents=True, exist_ok=True)
TRATADOS_DIR.mkdir(parents=True, exist_ok=True)
DOC_DIR.mkdir(parents=True, exist_ok=True)

# Configuracoes de requisicao
HEADERS = {
    "User-Agent": "TrabalhoAcademico-CienciaDeDados/1.0 (contato@exemplo.com)"
}
TIMEOUT = 30  # segundos
PAUSA = 1.5   # segundos entre requisicoes

print("Bibliotecas carregadas e pastas configuradas.")


ModuleNotFoundError: No module named 'pandas'

## 5. Fonte 1 — IBGE/SIDRA (API)

### 5.1 Descrição da fonte

O Sistema IBGE de Recuperação Automática (SIDRA) é uma plataforma do Instituto Brasileiro de Geografia e Estatística (IBGE) que disponibiliza dados estatísticos oficiais do Brasil. Para este trabalho, será utilizada a API do SIDRA para adquirir dados relacionados ao Índice Nacional de Preços ao Consumidor Amplo (IPCA), com foco nos preços e na inflação dos alimentos. Esses dados serão utilizados para analisar a evolução dos preços ao longo do período estudado e posteriormente integrados aos dados obtidos por Web Scraping.


### 5.2 Definição dos dados

A coleta da Fonte 1 é implementada no notebook **`notebook_ipca_alimentacao_1994_2026.py`**
(formato Jupyter `# %%`), que executa a estratégia descrita abaixo.

**Limitação da Tabela 61:** a Tabela 61 (IPCA - Peso mensal, para o índice geral, grupos, subgrupos, itens e subitens de produtos e serviços) cobre apenas **janeiro/1991 a julho/1999** e não possui a classificação `c315`. Para cobrir o período do projeto (1994 a 2026+) foi necessário pesquisar no SIDRA outras tabelas do IPCA com o **mesmo padrão** de classificação ("Geral, grupo, subgrupo, item e subitem") e concatená-las:

| Tabela | Período | Classificação | Grupo "1.Alimentação e bebidas" |
|---|---|---|---|
| 58 | jan/1991 – jul/1999 | c72 | 1313 |
| 655 | ago/1999 – jun/2006 | c315 | 7170 |
| 2938 | jul/2006 – dez/2011 | c315 | 7170 |
| 1419 | jan/2012 – dez/2019 | c315 | 7170 |
| 7060 | jan/2020 – atual | c315 | 7170 |

- **Indicador:** Índice Nacional de Preços ao Consumidor Amplo (IPCA);
- **Variável principal:** IPCA - Variação mensal (%) do grupo "1.Alimentação e bebidas" (a "cesta básica");
- **Cobertura:** janeiro/1994 até o mês mais recente disponível (agosto/2026 na coleta);
- **Dimensão temporal:** mês e ano de referência;
- **Unidade de observação:** grupo "1.Alimentação e bebidas" observado em determinado mês e ano;
- **Finalidade na pesquisa:** representar a inflação dos alimentos para comparar com os períodos de governos e ciclos eleitorais.

Como a série usa **variações mensais** (e não pesos), a troca de estrutura de ponderação entre eras não cria descontinuidade na base integrada.

**Fontes da definição:** [Tabela 58 — IPCA no SIDRA](https://sidra.ibge.gov.br/Tabela/58), [Tabela 655](https://sidra.ibge.gov.br/Tabela/655), [Tabela 2938](https://sidra.ibge.gov.br/Tabela/2938), [Tabela 1419](https://sidra.ibge.gov.br/Tabela/1419) e [Tabela 7060](https://sidra.ibge.gov.br/Tabela/7060).

### 5.3 URL da API

- `https://apisidra.ibge.gov.br/values/t/58/n1/all/v/all/p/all/c72/1313` *(jan/1991 – jul/1999)*
- `https://apisidra.ibge.gov.br/values/t/655/n1/all/v/all/p/all/c315/7170` *(ago/1999 – jun/2006)*
- `https://apisidra.ibge.gov.br/values/t/2938/n1/all/v/all/p/all/c315/7170` *(jul/2006 – dez/2011)*
- `https://apisidra.ibge.gov.br/values/t/1419/n1/all/v/all/p/all/c315/7170` *(jan/2012 – dez/2019)*
- `https://apisidra.ibge.gov.br/values/t/7060/n1/all/v/all/p/all/c315/7170` *(jan/2020 – atual)*



In [ ]:
### 5.3 Teste da requisição

# Teste com URL corrigida (Tabela 58, grupo Alimentação e bebidas)
url = "https://apisidra.ibge.gov.br/values/t/58/n1/all/v/all/p/last%201/c72/1313"

resposta = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
print("HTTP", resposta.status_code)

if resposta.status_code == 200:
    print("Requisição realizada com sucesso.")
    ultimo = resposta.json()[-1]
    print(ultimo["D3N"], "->", ultimo["D4N"], ":", ultimo["V"], "%")
else:
    print("Erro na requisição:", resposta.status_code)

In [ ]:
# 5.5 Coleta completa
#
# Série contínua da variação mensal do grupo "1.Alimentação e bebidas" (1994-2026+).
# Estratégia: concatena as tabelas IPCA de mesmo padrão (58, 655, 2938, 1419, 7060).
# Implementação completa e comentada: notebook_ipca_alimentacao_1994_2026.py

TABELAS_IPCA = [
    {"id": 58,   "classificacao": 72,  "categoria": 1313, "era": "1991-1999"},
    {"id": 655,  "classificacao": 315, "categoria": 7170, "era": "1999-2006"},
    {"id": 2938, "classificacao": 315, "categoria": 7170, "era": "2006-2011"},
    {"id": 1419, "classificacao": 315, "categoria": 7170, "era": "2012-2019"},
    {"id": 7060, "classificacao": 315, "categoria": 7170, "era": "2020-atual"},
]

API = "https://apisidra.ibge.gov.br/values"

def coletar_grupo_alimentacao(t):
    """GET do grupo Alimentação e bebidas de uma tabela SIDRA."""
    url = f"{API}/t/{t['id']}/n1/all/v/all/p/all/c{t['classificacao']}/{t['categoria']}"
    r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    if r.status_code != 200:
        print(f"t/{t['id']}: HTTP {r.status_code} (ERRO)")
        return None
    dados = r.json()
    df = pd.DataFrame(dados[1:])  # linha 0 é o cabeçalho
    df = df[df["D2N"] == "IPCA - Variação mensal"].copy()
    df["valor"] = pd.to_numeric(df["V"], errors="coerce")
    df["mes"] = df["D3C"].astype(str)
    df["data"] = pd.to_datetime(df["mes"], format="%Y%m")
    df["tabela"] = t["id"]
    print(f"t/{t['id']}: HTTP {r.status_code}, {len(df)} meses")
    return df[["data", "mes", "D4N", "valor", "tabela"]]

frames = []
for t in TABELAS_IPCA:
    df = coletar_grupo_alimentacao(t)
    if df is not None:
        frames.append(df)
    time.sleep(PAUSA)

ipca_alimentacao = pd.concat(frames, ignore_index=True)
ipca_alimentacao["grupo"] = ipca_alimentacao["D4N"]
ipca_alimentacao = ipca_alimentacao[ipca_alimentacao["mes"] >= "199401"]
ipca_alimentacao = ipca_alimentacao.drop_duplicates(subset=["mes"]).sort_values("data")
print(f"Série integrada: {len(ipca_alimentacao)} meses | "
      f"{ipca_alimentacao['data'].min().strftime('%Y-%m')} a "
      f"{ipca_alimentacao['data'].max().strftime('%Y-%m')}")
ipca_alimentacao.head()

In [ ]:
# 5.6 Salvamento dos dados brutos
#
# Cada resposta bruta da API é preservada em dados_brutos/ antes de qualquer
# limpeza, e a proveniência de cada requisição é registrada em CSV.

data_hora_coleta = datetime.now(timezone.utc)
stamp = data_hora_coleta.strftime("%Y%m%d_%H%M%S")
proveniencia = []

for t in TABELAS_IPCA:
    url = f"{API}/t/{t['id']}/n1/all/v/all/p/all/c{t['classificacao']}/{t['categoria']}"
    r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)

    prov = {
        "fonte_id": f"SIDRA-t{t['id']}",
        "url": url,
        "era": t["era"],
        "data_hora_coleta": data_hora_coleta.isoformat(),
        "metodo": "API GET",
        "status_http": r.status_code,
        "arquivo_bruto": None,
        "quantidade_registros": None,
        "observacoes": "Grupo 1.Alimentação e bebidas; variável: IPCA - Variação mensal",
    }
    if r.status_code == 200:
        arquivo = BRUTOS_DIR / f"sidra_t{t['id']}_alimentacao_{stamp}.json"
        with open(arquivo, "w", encoding="utf-8") as f:
            json.dump(r.json(), f, ensure_ascii=False, indent=2)
        prov["arquivo_bruto"] = str(arquivo)
        prov["quantidade_registros"] = len(r.json()) - 1
    proveniencia.append(prov)
    time.sleep(PAUSA)

proveniencia_df = pd.DataFrame(proveniencia)
prov_file = BRUTOS_DIR / f"proveniencia_sidra_{stamp}.csv"
proveniencia_df.to_csv(prov_file, index=False, encoding="utf-8")
print("Respostas brutas salvas e proveniência registrada.")
print("Proveniência:", prov_file)
proveniencia_df[["fonte_id", "era", "status_http", "quantidade_registros", "arquivo_bruto"]]

## 6. Fonte 2 — DIEESE (Web Scraping)

### 6.1 Descrição da fonte

O DIEESE (Departamento Intersindical de Estatística e Estudos Socioeconômicos) publica mensalmente a **Pesquisa Nacional da Cesta Básica**, que registra o valor da cesta básica de alimentos em diversas capitais brasileiras, além do tempo de trabalho necessário para comprá-la e as variações mensais, anuais e em 12 meses.

A coleta dos boletins mensais é realizada a partir da página de índice de boletins anteriores do DIEESE, onde cada boletim está disponível como PDF (ou, em alguns casos, como página HTML intermediária que redireciona para o PDF). Todo o processo — coleta, limpeza de arquivos HTML "casca", extração de tabelas dos PDFs e correção manual de meses problemáticos — é documentado de forma reprodutível no notebook **`coletaDados.ipynb`**.

**Fonte dos dados:** [DIEESE — Análise da Cesta Básica](https://www.dieese.org.br/analisecestabasica/analiseCestaBasicaAnteriores.html)

> ⚠️ Este notebook faz requisições HTTP reais ao site do DIEESE e respeita um intervalo de 10 segundos entre downloads (conforme o `Crawl-delay` do `robots.txt` do site). Rodar o notebook do zero, portanto, pode levar bastante tempo dependendo de quantos boletins ainda não estiverem salvos localmente em `projeto/dados_brutos/dieese/`.

### 6.2 URL

- Página de índice de boletins: `https://www.dieese.org.br/analisecestabasica/analiseCestaBasicaAnteriores.html`

### 6.3 Estrutura HTML

A página de índice contém uma lista de links para boletins mensais, identificados pelo padrão `cestabasica` seguido ou precedido por 6 dígitos (formato `AAMMES`, ex: `202001`). Os boletins podem ser:

- **Arquivos PDF diretos**: baixados e salvos como `{aammes}.pdf`
- **Páginas HTML "casca"**: páginas intermediárias que apenas exibem um link para o PDF real. São identificadas pelo texto `"Resultados Mensais de"` e possuem o link do PDF real embutido em um tag `<a href="...cestabasica.pdf">`. Após baixar o PDF, o `.html` casca é removido.

### 6.4 Scraping

O processo de coleta segue os passos abaixo (implementação completa em `coletaDados.ipynb`):

1. Acessar a página de índice e extrair todos os links de boletins mensais via expressão regular `(cestabasica\d{6}|\d{6}cestabasica)`
2. Para cada boletim:
   - Se o arquivo (`.html` ou `.pdf`) já existir localmente, registrar no log de proveniência com a data de modificação do arquivo existente (**sem** re-baixar)
   - Caso contrário, baixar o arquivo via HTTP e salvar em `projeto/dados_brutos/dieese/{aammes}.{extensao}`
   - Registrar a proveniência (URL, status HTTP, formato, data/hora) em `projeto/dados_brutos/dieese/provenance_log.csv`
   - Respeitar o intervalo de **10 segundos** entre requisições (Crawl-delay do `robots.txt`)

```python
BASE = "https://www.dieese.org.br"
INDEX_URL = f"{BASE}/analisecestabasica/analiseCestaBasicaAnteriores.html"
HEADERS = {"User-Agent": "Projeto-UFAM/1.0"}

Path("dados_brutos/dieese").mkdir(parents=True, exist_ok=True)
caminho_log = "dados_brutos/dieese/provenance_log.csv"
campos = ["aammes", "titulo", "url", "status_http", "formato", "coletado_em"]

arquivo_novo = not Path(caminho_log).exists()
arquivo_log = open(caminho_log, "a", newline="", encoding="utf-8")
escritor = csv.DictWriter(arquivo_log, fieldnames=campos)
if arquivo_novo:
    escritor.writeheader()

# Baixar página de índice e extrair links de boletins
resposta = requests.get(INDEX_URL, headers=HEADERS, timeout=30)
sopa = BeautifulSoup(resposta.text, "html.parser")
padrao = re.compile(r"(cestabasica\d{6}|\d{6}cestabasica)", re.IGNORECASE)
boletins = [(a.get_text(strip=True), a["href"]) for a in sopa.find_all("a", href=True) if padrao.search(a["href"])]

for titulo, href in boletins:
    url = href if href.startswith("http") else BASE + href
    extensao = url.split(".")[-1]
    aammes = re.search(r"(\d{6})", url).group(1)
    # Lógica de download e registro de proveniência...
```

### 6.5 Salvamento dos dados brutos

- **Arquivos baixados**: `projeto/dados_brutos/dieese/{aammes}.pdf` e/ou `{aammes}.html`
- **Log de proveniência**: `projeto/dados_brutos/dieese/provenance_log.csv` (campos: aammes, titulo, url, status_http, formato, coletado_em)
- **Formato**: PDF (principal) e HTML (intermediário/casca)
- **Observação**: arquivos já existentes não são re-baixados; apenas registrados no log com data de modificação do arquivo em disco



In [ ]:
# 6.4 Scraping — coleta dos boletins mensais
# Acessa a página de índice, identifica links de boletins e baixa cada um,
# respeitando o intervalo de 10s entre requisições.

BASE = "https://www.dieese.org.br"
INDEX_URL = f"{BASE}/analisecestabasica/analiseCestaBasicaAnteriores.html"
HEADERS = {"User-Agent": "Projeto-UFAM/1.0"}
pasta = BRUTOS_DIR / "dieese"
pasta.mkdir(parents=True, exist_ok=True)

caminho_log = pasta / "provenance_log.csv"
campos = ["aammes", "titulo", "url", "status_http", "formato", "coletado_em"]

arquivo_novo = not caminho_log.exists()
arquivo_log = open(caminho_log, "a", newline="", encoding="utf-8")
escritor = csv.DictWriter(arquivo_log, fieldnames=campos)
if arquivo_novo:
    escritor.writeheader()

# Baixa a página de índice e extrai links de boletins mensais
resposta = requests.get(INDEX_URL, headers=HEADERS, timeout=30)
sopa = BeautifulSoup(resposta.text, "html.parser")
padrao = re.compile(r"(cestabasica\d{6}|\d{6}cestabasica)", re.IGNORECASE)
boletins = [(a.get_text(strip=True), a["href"]) for a in sopa.find_all("a", href=True) if padrao.search(a["href"])]
print("Total de boletins encontrados:", len(boletins))

for titulo, href in boletins:
    url = href if href.startswith("http") else BASE + href
    extensao = url.split(".")[-1]
    aammes = re.search(r"(\d{6})", url).group(1)
    caminho_html = pasta / f"{aammes}.html"
    caminho_pdf = pasta / f"{aammes}.pdf"

    if caminho_html.exists() or caminho_pdf.exists():
        existente = caminho_html if caminho_html.exists() else caminho_pdf
        hora = datetime.fromtimestamp(existente.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
        escritor.writerow({"aammes": aammes, "titulo": titulo, "url": url, "status_http": 200,
                           "formato": existente.suffix[1:], "coletado_em": hora + " (recuperado, já existia)"})
        arquivo_log.flush()
        print(aammes, "-> já existia, registrado no log")
        continue

    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code == 200:
            with open(pasta / f"{aammes}.{extensao}", "wb") as f:
                f.write(r.content)
            print(aammes, "-> salvo como", extensao)
        else:
            print(aammes, "-> falhou, status", r.status_code)
        escritor.writerow({"aammes": aammes, "titulo": titulo, "url": url,
                           "status_http": r.status_code, "formato": extensao,
                           "coletado_em": time.strftime("%Y-%m-%d %H:%M:%S")})
    except requests.exceptions.RequestException as erro:
        print(aammes, "-> erro, pulando:", erro)
        escritor.writerow({"aammes": aammes, "titulo": titulo, "url": url,
                           "status_http": "erro", "formato": extensao,
                           "coletado_em": time.strftime("%Y-%m-%d %H:%M:%S")})

    arquivo_log.flush()
    time.sleep(10)

arquivo_log.close()
print("Coleta finalizada.")


In [ ]:
# 6.5 Salvamento dos dados brutos
# Arquivos brutos e proveniência já foram salvos durante o scraping (6.4).
# Verificação dos arquivos salvos:

print(f"Arquivos em {pasta}:")
for f in sorted(pasta.iterdir()):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")

print(f"\nLog de proveniência: {caminho_log}")
if caminho_log.exists():
    with open(caminho_log, "r", encoding="utf-8") as lf:
        print(lf.read())


## 7. Fonte 3 — TSE/Dados Abertos (API)

### 7.1 Descrição da fonte

O Portal de Dados Abertos do Tribunal Superior Eleitoral (TSE) é uma plataforma que disponibiliza dados eleitorais oficiais do Brasil. Para este trabalho, serão utilizadas duas APIs do TSE:

1. **API CKAN** — para consulta de metadados e obtenção de URLs de recursos (datasets):
   `https://dadosabertos.tse.jus.br/api/3/action`
   - Endpoint `package_search`: busca datasets por termo (ex: `candidatos`, `resultados`).
   - Endpoint `package_show`: retorna metadados completos de um dataset, incluindo os recursos (arquivos) disponíveis.
   - Endpoint `resource_show`: retorna detalhes de um recurso específico, incluindo a URL de download.
   - Todos os recursos retornam dados no formato JSON, seguindo o padrão CKAN.

2. **API DivulgaCandContas (REST)** — para acesso em tempo real a informações sobre eleições, candidatos e suas respectivas informações:
   `https://divulgacandcontas.tse.jus.br/divulga/rest/v1`
   - Endpoint `/eleicao/ordinarias`: lista todas as eleições ordinárias disponíveis, com IDs, anos, tipos e datas.
   - Endpoint `/candidatura/listar/{ano}/{municipio_cod}/{eleicao_id}/{cargo_cod}/candidatos`: lista candidatos de um cargo específico em uma eleição.
   - Endpoint `/candidatura/buscar/{ano}/{municipio_cod}/{eleicao_id}/candidato/{candidato_id}`: retorna detalhes completos de um candidato.

Para o tema deste projeto, os dados relevantes são:

- **Datasets de Candidatos** (`candidatos-{ano}`): contêm informações sobre candidatos, incluindo nome, partido (sigla), cargo, ano e UF. Disponíveis para os anos: 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022.
- **Datasets de Resultados** (`resultados-{ano}`): contêm resultados eleitorais detalhados por município e zona. Disponíveis a partir de 1994.

Esses dados permitem mapear:

- os governos presidenciais e estaduais ao longo do tempo (períodos de 1994 a 2022);
- os ciclos eleitorais (anos de eleições gerais e municipais);
- os espectros partidários dos governantes (siglas de partidos e coligações).

Como os dados são disponibilizados em arquivos ZIP contendo CSVs, a coleta envolve a API CKAN para obter os metadados e URLs, seguida do download e descompressão dos arquivos.

**Fonte da definição:** [Portal de Dados Abertos do TSE](https://dadosabertos.tse.jus.br/) e [DivulgaCandContas REST API](https://divulgacandcontas.tse.jus.br/divulga/rest/v1).

### 7.2 URL da API

**API CKAN (Datasets):**

`https://dadosabertos.tse.jus.br/api/3/action`

**API DivulgaCandContas (Tempo real):**

`https://divulgacandcontas.tse.jus.br/divulga/rest/v1`

### 7.3 Endpoints e parâmetros

| Endpoint | Método | Parâmetros | Descrição |
|---|---|---|---|
| `package_search` | GET | `q=<termo>`, `rows=<n>` | Busca datasets por termo |
| `package_show` | GET | `id=<slug>` | Metadados e recursos de um dataset |
| `resource_show` | GET | `id=<resource_id>` | URL e metadados de um recurso |
| `/eleicao/ordinarias` | GET | — | Lista eleições ordinárias |
| `/candidatura/listar/{ano}/{municipio_cod}/{eleicao_id}/{cargo_cod}/candidatos` | GET | — | Lista candidatos de um cargo |

### 7.4 Dados para este projeto

- **Cargo Presidencial (código 1)**: anos eleitorais 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022.
- **Cargo Governador (código 3)**: mesmos anos eleitorais.
- **Chave de integração**: `ano` (ano eleitoral) e `sigla_partido` (partido do candidato).
- **Licença**: Creative Commons Atribuição (CC-BY).



In [ ]:
### 7.5 Teste da requisição
import requests

# API CKAN - buscar datasets de candidatos
CKAN_URL = "https://dadosabertos.tse.jus.br/api/3/action"
DIVULGA_URL = "https://divulgacandcontas.tse.jus.br/divulga/rest/v1"

# Teste 1: package_search por 'candidatos'
r = requests.get(
    f"{CKAN_URL}/package_search",
    params={"q": "candidatos", "rows": 20},
    headers=HEADERS,
    timeout=TIMEOUT,
)
print(f"package_search candidatos: HTTP {r.status_code}")

# Teste 2: package_show para 'candidatos-1994'
r2 = requests.get(
    f"{CKAN_URL}/package_show",
    params={"id": "candidatos-1994"},
    headers=HEADERS,
    timeout=TIMEOUT,
)
print(f"package_show candidatos-1994: HTTP {r2.status_code}")
if r2.status_code == 200:
    ds = r2.json()["result"]
    print(f"  Título: {ds.get('title')}")
    print(f"  Licença: {ds.get('license_id')}")
    print(f"  Recursos: {len(ds.get('resources', []))}")
    for res in ds.get("resources", [])[:3]:
        print(f"    - {res.get('name')}: {res.get('url')[:80]}...")

# Teste 3: DivulgaCandContas - eleicoes ordinarias
r3 = requests.get(
    f"{DIVULGA_URL}/eleicao/ordinarias",
    headers=HEADERS,
    timeout=TIMEOUT,
)
print(f"\neleicao/ordinarias: HTTP {r3.status_code}")
if r3.status_code == 200:
    eleicoes = r3.json()
    print(f"  Eleições encontradas: {len(eleicoes)}")
    for e in eleicoes[:5]:
        print(f"    - {e.get('ano')}: {e.get('nomeEleicao')} (ID: {e.get('id')}, tipo: {e.get('tipoAbrangencia')})")

# Teste 4: package_search por 'resultados'
r4 = requests.get(
    f"{CKAN_URL}/package_search",
    params={"q": "resultados", "rows": 20},
    headers=HEADERS,
    timeout=TIMEOUT,
)
print(f"\npackage_search resultados: HTTP {r4.status_code}")
if r4.status_code == 200:
    results = r4.json()["result"]["results"]
    for item in results[:8]:
        print(f"  - {item.get('name')}")


In [ ]:
# 7.6 Coleta completa
#
# Anos eleitorais de interesse (presidenciais e gubernatoriais desde 1994):
# 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022.
#
# Estratégia:
# 1. Consultar a API CKAN para obter os recursos (resources) de download (ZIPs)
#    dos datasets 'candidatos-{ano}' e 'resultados-{ano}'.
# 2. Baixar os arquivos ZIP brutos conservando-os em dados_brutos/tse/.
# 3. Registrar a data/hora da coleta, status HTTP e metadados.

# Anos de eleicoes gerais (presidenciais e gubernatoriais)
ANOS_ELEICAO = [1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]

# Pasta para arquivos brutos do TSE
TSE_BRUTOS_DIR = BRUTOS_DIR / "tse"
TSE_BRUTOS_DIR.mkdir(parents=True, exist_ok=True)

# Data/hora da coleta (UTC)
data_hora_coleta = datetime.now(timezone.utc)
print(f"Coleta iniciada em: {data_hora_coleta.isoformat()}")

# Lista para registrar a proveniencia de cada recurso
proveniencia_tse = []

# Função para obter recursos (resources) de um dataset via API CKAN
def obter_recursos_dataset(slug: str) -> list:
    """Usa a API CKAN para obter a lista de recursos de um dataset."""
    try:
        r = requests.get(
            f"{CKAN_URL}/package_show",
            params={"id": slug},
            headers=HEADERS,
            timeout=TIMEOUT,
        )
        if r.status_code == 200:
            return r.json().get("result", {}).get("resources", [])
        print(f"  Erro HTTP {r.status_code} para dataset '{slug}'")
    except Exception as e:
        print(f"  Erro de conexão para '{slug}': {e}")
    return []

# Função para baixar um arquivo e salvar o bruto
def baixar_arquivo(url: str, destino: Path) -> dict:
    """Baixa um arquivo da URL e salva em destino. Retorna dict com proveniencia."""
    info = {
        "url": url,
        "status_http": None,
        "arquivo": str(destino),
        "tamanho_bytes": None,
    }
    try:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        info["status_http"] = r.status_code
        if r.status_code == 200:
            destino.parent.mkdir(parents=True, exist_ok=True)
            with open(destino, "wb") as f:
                f.write(r.content)
            info["tamanho_bytes"] = len(r.content)
        else:
            print(f"  Erro HTTP {r.status_code} ao baixar: {url[:80]}...")
    except Exception as e:
        info["status_http"] = "erro"
        print(f"  Erro de conexão: {e}")
    return info

# Coletar datasets de candidatos e resultados para cada ano
for ano in ANOS_ELEICAO:
    print(f"\nProcessando ano: {ano}")

    # --- Candidatos ---
    slug_cand = f"candidatos-{ano}"
    recursos_cand = obter_recursos_dataset(slug_cand)
    for rec in recursos_cand:
        if rec.get("format", "").upper() == "CSV":
            nome_arquivo = f"{rec['name'].replace(' ', '_')}_{ano}.zip"
            destino = TSE_BRUTOS_DIR / nome_arquivo
            info = baixar_arquivo(rec["url"], destino)
            proveniencia_tse.append({
                "fonte_id": "TSE-CKAN",
                "dataset": slug_cand,
                "recurso": rec["name"],
                "data_hora_coleta": data_hora_coleta.isoformat(),
                **info,
            })
            time.sleep(PAUSA)

    # --- Resultados (se disponível) ---
    slug_res = f"resultados-{ano}"
    recursos_res = obter_recursos_dataset(slug_res)
    for rec in recursos_res:
        if rec.get("format", "").upper() == "CSV":
            nome_arquivo = f"{rec['name'].replace(' ', '_')}_{ano}.zip"
            destino = TSE_BRUTOS_DIR / nome_arquivo
            info = baixar_arquivo(rec["url"], destino)
            proveniencia_tse.append({
                "fonte_id": "TSE-CKAN",
                "dataset": slug_res,
                "recurso": rec["name"],
                "data_hora_coleta": data_hora_coleta.isoformat(),
                **info,
            })
            time.sleep(PAUSA)

print(f"\nColeta concluída. Recursos baixados: {len(proveniencia_tse)}")
for p in proveniencia_tse:
    print(f"  {p['dataset']}/{p['recurso']}: HTTP {p['status_http']}")


In [ ]:
# 7.7 Salvamento dos dados brutos
#
# Os arquivos ZIP ja foram salvos em dados_brutos/tse/ durante a coleta.
# Aqui salvamos tambem a resposta bruta da API CKAN (JSON) e o registro
# de proveniencia consolidado.

# Salvar resposta bruta da API CKAN (package_show para um dataset exemplo)
ds_exemplo = requests.get(
    f"{CKAN_URL}/package_show",
    params={"id": "candidatos-2022"},
    headers=HEADERS,
    timeout=TIMEOUT,
)
raw_api_file = BRUTOS_DIR / f"tse_ckan_api_resposta_{data_hora_coleta.strftime('%Y%m%d_%H%M%S')}.json"
with open(raw_api_file, "w", encoding="utf-8") as f:
    json.dump(ds_exemplo.json(), f, ensure_ascii=False, indent=2)
print(f"Resposta bruta da API CKAN salva em: {raw_api_file}")

# Salvar registro de proveniencia do TSE
proveniencia_df = pd.DataFrame(proveniencia_tse)
proveniencia_file = BRUTOS_DIR / f"proveniencia_tse_{data_hora_coleta.strftime('%Y%m%d_%H%M%S')}.csv"
proveniencia_df.to_csv(proveniencia_file, index=False, encoding="utf-8")
print(f"Registro de proveniencia salvo em: {proveniencia_file}")
proveniencia_df


### 8.1 Tratamento da Fonte 1 — IBGE/SIDRA

A série do IPCA para o grupo "Alimentação e bebidas" foi construída a partir de múltiplas tabelas do SIDRA, em vez de uma única tabela contínua, porque cada tabela cobre um período distinto. As principais tabelas usadas foram:

- Tabela 58: jan/1991 a jul/1999
- Tabela 655: ago/1999 a jun/2006
- Tabela 2938: jul/2006 a dez/2011
- Tabela 1419: jan/2012 a dez/2019
- Tabela 7060: jan/2020 até o mês mais recente

A lógica do tratamento foi:

1. Consultar a API do SIDRA para cada tabela;
2. Salvar o JSON bruto em `projeto/dados_brutos/ibge_sidra/` antes de qualquer transformação;
3. Filtrar apenas a variável `IPCA - Variação mensal` e o grupo `Alimentação e bebidas`;
4. Normalizar colunas como `mes`, `grupo`, `variavel`, `valor`, `tabela`;
5. Concatenar as tabelas em uma única série temporal;
6. Restringir a série a partir de `199401`;
7. Eliminar duplicatas por mês e grupo;
8. Verificar continuidade temporal e registrar meses faltantes;
9. Exportar a base tratada em `projeto/dados_tratados/ipca_alimentacao_tratada.csv`.

Essa etapa preserva a série mensal de inflação dos alimentos como variável de contexto econômico para a integração com a cesta básica e com o cenário eleitoral.

### 8.2 Tratamento da Fonte 2 — DIEESE

O tratamento dos dados do DIEESE segue três etapas principais e foi validado no projeto para exportação em `projeto/dados_tratados/cesta_basica_dieese_tratada.csv`.

#### 8.2.1 Limpeza dos arquivos HTML ("cascas")



Alguns boletins baixados são páginas HTML intermediárias ("cascas") que apenas redirecionam para o PDF real. Esta etapa:Esse conjunto de etapas torna as três fontes comparáveis e compatíveis para a integração analítica final.



1. Varre todos os `.html` em `projeto/dados_brutos/dieese/`- registro de dados ausentes e validade temporal por mês.

2. Identifica cascas pelo texto `"Resultados Mensais de"`- sinalização de inconsistências por flags (`flag_capital_variante`, `flag_diverge_var_mensal`, `flag_conflito_capital_mes`);

3. Extrai o link do PDF real e baixa o PDF- remoção de duplicatas exatas;

4. Substitui o `.html` pelo `.pdf` correspondente- coerção de colunas numéricas para `float`/`Int64` quando apropriado;

5. Mantém os HTMLs que já eram conteúdo real- conversão do campo `tempo` para horas decimais (`tempo_em_horas`);

- nomes de capitais normalizados para 27 nomes canônicos e `sigla_uf`;

#### 8.2.2 Extração das tabelas dos PDFs- `mes`, `ano`, `data_referencia` e `mes_num` padronizados para a base DIEESE;



Cada boletim PDF contém uma tabela com uma linha por capital e campos como `valor`, `var_mensal`, `pct_sal`, `tempo`, `var_ano` e `var_12`. A extração foi feita com `pdfplumber` por posicionamento relativo dos campos, com robustez para variações de layout entre meses.Após o tratamento individual, as bases passaram por uma padronização comum para integração:



Exceção: o boletim de **julho/2005** (`200507`) teve formatação divergente e foi transcrito manualmente.### 8.4 Padronização final das bases



#### 8.2.3 Correção manual e exportaçãoA tabela derivada preserva a relação entre eleição, cargo, UF, partido e coligação, que será usada para associar cada observação da base econômica ao governo vigente em cada período.



O boletim de **julho/2005** (`200507`) foi ajustado manualmente a partir do PDF oficial e inserido na base. O resultado final do tratamento do DIEESE foi exportado para:8. Exportar em `projeto/dados_tratados/tse_derivada_presidentes_governadores.csv` e `.parquet`.

7. Consolidar os anos em uma única tabela derivada;

- `projeto/dados_tratados/cesta_basica_dieese_tratada.csv`6. Converter campos numéricos para tipos adequados (`ANO_ELEICAO`, `CD_CARGO`, `NR_PARTIDO`, etc.);

- `projeto/dados_tratados/cesta_basica_dieese_tratada.parquet`5. Selecionar colunas de interesse, sem dados pessoais sensíveis;

4. Manter somente candidatos eleitos (`CD_SIT_TOT_TURNO = 1`);

Ao final, a base foi padronizada com a chave de mês, capital e UF, além de colunas numéricas e flags de consistência.3. Filtrar apenas os cargos de interesse: presidente (`CD_CARGO = 1`) e governador (`CD_CARGO = 3`);

2. Ler os CSVs internos do ZIP com separador `;` e codificação `latin-1`;

### 8.3 Tratamento da Fonte 3 — TSE1. Carregar os arquivos ZIP dos candidatos por ano eleitoral: `1994`, `1998`, `2002`, `2006`, `2010`, `2014`, `2018`, `2022`;



A base do TSE foi tratada como uma tabela derivada de candidatos eleitos em anos eleitorais. O objetivo foi reduzir o conjunto bruto para o escopo analítico do projeto, preservando somente informações relevantes para contextualizar o cenário político por período de governo.Os passos do tratamento foram:


## 9. Integração das fontes

### 9.1 Chave de integração

A integração do projeto usa duas chaves principais:

| Fonte | Método | Chave de integração principal |
|---|---|---|
| Fonte 1 — IBGE/SIDRA | API | `mes` |
| Fonte 2 — DIEESE | scraping + PDF | `mes`, `capital`, `sigla_uf` |
| Fonte 3 — TSE/Dados Abertos | API | `ano_eleitoral_referencia`, `sigla_uf` e `ano_eleitoral_presidente_referencia` |

A base DIEESE é integrada ao IPCA por `mes`, e depois o contexto eleitoral do TSE é anexado por ano eleitoral e UF. Isso preserva uma linha por capital/mês e adiciona o governo vigente em cada período.

### 9.2 Junção

A junção foi realizada em três etapas:

1. Carregamento da base tratada do DIEESE em `projeto/dados_tratados/cesta_basica_dieese_tratada.csv`.
2. Mesclagem com a série do IPCA tratada em `projeto/dados_tratados/ipca_alimentacao_tratada.csv`, usando a chave `mes`.
3. Anexo dos dados do TSE derivado em `projeto/dados_tratados/tse_derivada_presidentes_governadores.csv`, relacionando o governo vigente por `ano eleitoral` e `UF`.

A saída intermediária final é a base integrada em `projeto/dados_tratados/base_integrada.csv`.

### 9.3 Verificação

A validação inclui:

- contagem de linhas e colunas;
- verificação de correspondência do IPCA por mês;
- checagem de linhas sem associação eleitoral;
- controle de nulos e consistência das colunas principais.

Resultado observado: `base_integrada.csv` contém 4.757 registros e 35 colunas.


In [ ]:
# 9.4 Integração e verificação
from pathlib import Path
import pandas as pd

TRATADOS_DIR = Path("projeto/dados_tratados")
base_dieese = pd.read_csv(TRATADOS_DIR / "cesta_basica_dieese_tratada.csv")
ipca = pd.read_csv(TRATADOS_DIR / "ipca_alimentacao_tratada.csv")
tse = pd.read_csv(TRATADOS_DIR / "tse_derivada_presidentes_governadores.csv", encoding="utf-8-sig")

ipca_join = ipca[["mes", "grupo", "variavel", "valor", "tabela"]].rename(
    columns={
        "grupo": "ipca_grupo",
        "variavel": "ipca_variavel",
        "valor": "ipca_var_mensal_pct",
        "tabela": "ipca_tabela_sidra",
    }
)

base_integrada = base_dieese.merge(ipca_join, on="mes", how="left", validate="many_to_one")

anos_eleitorais = sorted(tse["ANO_ELEICAO"].dropna().astype(int).unique())
anos_presidenciais = sorted(tse.loc[tse["CD_CARGO"].eq(1), "ANO_ELEICAO"].dropna().astype(int).unique())
anos_governadores = sorted(tse.loc[tse["CD_CARGO"].eq(3), "ANO_ELEICAO"].dropna().astype(int).unique())


def ano_eleitoral_referencia(ano: int, anos_disponiveis: list[int]):
    return max((eleicao for eleicao in anos_disponiveis if eleicao <= ano), default=pd.NA)

base_integrada["ano_eleitoral_referencia"] = base_integrada["ano"].map(
    lambda ano: ano_eleitoral_referencia(ano, anos_eleitorais)
).astype("Int64")
base_integrada["ano_eleitoral_presidente_referencia"] = base_integrada["ano"].map(
    lambda ano: ano_eleitoral_referencia(ano, anos_presidenciais)
).astype("Int64")
base_integrada["ano_eleitoral_governador_referencia"] = base_integrada["ano"].map(
    lambda ano: ano_eleitoral_referencia(ano, anos_governadores)
).astype("Int64")

print("Linhas da base integrada:", base_integrada.shape[0])
print("Colunas da base integrada:", base_integrada.shape[1])
print("Linhas sem IPCA:", int(base_integrada["ipca_var_mensal_pct"].isna().sum()))
print("Linhas sem governador de referência:", int(base_integrada["tse_governador_nm_candidato"].isna().sum()))


## 10. Base final

### 10.1 Estrutura

A base final está em `projeto/dados_tratados/base_integrada.csv` e preserva uma linha por capital/mês. As principais variáveis incluem:

- `mes`, `data_referencia`, `ano`, `mes_num`
- `capital`, `sigla_uf`, `capital_bruto`
- `valor`, `var_mensal`, `pct_sal`, `tempo`, `tempo_em_horas`
- `ipca_var_mensal_pct`, `ipca_grupo`, `ipca_variavel`
- `ano_eleitoral_presidente_referencia`, `ano_eleitoral_governador_referencia`
- `tse_presidente_nm_candidato`, `tse_governador_nm_candidato`, `tse_presidente_sg_partido`, `tse_governador_sg_partido`

### 10.2 Quantidade de registros

Os resultados finais confirmados são:

- base DIEESE tratada: 4.757 linhas e 20 colunas
- série IPCA tratada: 392 linhas e 6 colunas
- base final integrada: 4.757 linhas e 35 colunas

### 10.3 Exportação

A base final é exportada em dois formatos:

- `projeto/dados_tratados/base_integrada.csv`
- `projeto/dados_tratados/base_integrada.parquet`

A exportação foi validada e os arquivos encontram-se presentes no diretório de dados tratados do projeto.


In [ ]:
# Exportação da base final
from pathlib import Path
import pandas as pd

TRATADOS_DIR = Path("projeto/dados_tratados")
base = pd.read_csv(TRATADOS_DIR / "base_integrada.csv")
base.to_csv(TRATADOS_DIR / "base_integrada.csv", index=False, encoding="utf-8-sig")
base.to_parquet(TRATADOS_DIR / "base_integrada.parquet", index=False)

print("Base final exportada:", TRATADOS_DIR / "base_integrada.csv")
print("Parquet exportado:", TRATADOS_DIR / "base_integrada.parquet")
print("Formato final:", base.shape)


## 11. Proveniência e observações

### 11.1 Registros de proveniência

Cada fonte mantém seu próprio log de proveniência:

| Fonte | Arquivo de proveniência | Descrição |
|---|---|---|
| Fonte 1 — IBGE/SIDRA | `projeto/dados_brutos/proveniencia/sidra_t{N}_alimentacao_{stamp}.json` + `proveniencia_sidra_{stamp}.csv` | URL, status HTTP, data/hora, tamanho |
| Fonte 2 — DIEESE | `projeto/dados_brutos/dieese/provenance_log.csv` | boletim, URL, status HTTP, formato, data/hora |
| Fonte 3 — TSE | `projeto/dados_brutos/proveniencia/proveniencia_tse.csv` | dataset, recurso, URL, status HTTP, tamanho |

### 11.2 Observações

- **DIEESE**: Coleta respeita Crawl-delay de 10s (robots.txt). Arquivos já existentes não são re-baixados. Boletim 201001 corrigido (URL mapeada incorretamente). Julho/2005 transcrito manualmente.
- **TSE**: CKAN API protegida por Akamai (intermitente HTTP 403, contornado via cookies). Arquivos ZIP reportados como CSV pela API. CDN downloads diretos confirmados.
- **IBGE/SIDRA**: API pública, sem proteção anti-bot. Múltiplas tabelas concatenadas para cobertura contínua (1994-2026+).
